# 4조 · 암 아형(SUBCLASS) 분류 · 최종 제출 코드

> **이 파일 하나로 학습부터 추론까지 실행돼요.** 통합 담당: ④ 박유정 · 최종 확인: ① 장성은
> 각 담당자는 자기 칸(【② 피처】【③ 모델】【④ 경량·앙상블】)에만 최종 코드를 넣어요. 칸 밖은 고치지 않아요.

## 제출 정보 (11/11 제출 전에 채우기)
| 항목 | 내용 |
|---|---|
| 팀 | 4조 (장성은, 홍시윤, 안주형, 박유정) |
| 개발 환경(OS) | Google Colab (Linux, Python 3.13.18), CPU |
| 라이브러리 버전 | numpy 2.4.6, pandas 2.2.3, scikit-learn 1.6.1, xgboost 3.4.1, scipy 1.16.3 |
| Random Seed | 42 (`SEED`) |
| 하이퍼파라미터 | 【③ 모델】 칸에 숫자로 기재 |
| 모델 파일 다운로드 링크 | (작성 예정) |
| 사전 학습 모델 | 사용 안 함 (사용 시 논문 링크와 출처 기재) |
| 외부 데이터 | 사용 안 함 |

## 데이터 누수 방지
- 모든 전처리·피처 선택·모델 학습은 train에만 fit 해요. test에는 transform과 predict만 해요.
- test의 통계, 분포, 행 개수, index는 쓰지 않아요.

## 1. 설정

In [ ]:
import os, random, platform
import numpy as np
import pandas as pd

DATA_DIR = os.environ.get("DATA_DIR", "/data")   # 제출 규정: 데이터 입출력 경로 /data
OUTPUT_PATH = os.path.join(DATA_DIR, "submission.csv")
SEED = 42
N_SPLITS = 5
RUN_CV = False          # True면 교차검증 점수도 다시 계산해요 (시간이 오래 걸림)
TARGET, ID_COL = "SUBCLASS", "ID"

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

## 2. 환경 정보 (출력값을 위 표에 옮겨 적어요)

In [ ]:
import sklearn, xgboost
print("OS:", platform.platform())
print("python:", platform.python_version())
print("numpy:", np.__version__, "| pandas:", pd.__version__,
      "| scikit-learn:", sklearn.__version__, "| xgboost:", xgboost.__version__)

## 3. 데이터 불러오기

In [ ]:
from sklearn.preprocessing import LabelEncoder

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
feature_cols = [c for c in train.columns if c not in (ID_COL, TARGET)]
assert list(test.columns) == [ID_COL] + feature_cols   # 형식 확인만

X = train[feature_cols]
X_test = test[feature_cols]
le = LabelEncoder().fit(train[TARGET])                  # train 정답에만 fit
y = le.transform(train[TARGET])
print("train", X.shape, "| test", X_test.shape, "| classes", len(le.classes_))

## 4. 【② 피처】 전처리·피처 · 담당 ② 홍시윤
`src/preprocessing.py`, `src/features.py`의 최종 함수를 그대로 붙여 넣어요 (화요일 12시 확정본).

In [ ]:
# ===== 【② 피처】 시작 =====
# 출처: src/preprocessing.py, src/features.py (dev 기준) + 02_features "4-2. E" 피처 구성
import re
from sklearn.preprocessing import FunctionTransformer

WT = "WT"


def binarize_mutations(X: pd.DataFrame) -> pd.DataFrame:
    """[팀 기본 피처] WT 또는 결측 = 0, 변이 = 1."""
    return (X.notna() & (X != WT)).astype(np.int8)


def mutation_count(X: pd.DataFrame) -> pd.Series:
    """환자별 변이가 있는 유전자 개수."""
    return binarize_mutations(X).sum(axis=1).rename("n_mutated_genes")


def mutation_type_counts(X):
    """환자별 변이 유형 개수 (n_missense, n_synonymous, n_nonsense, n_frameshift, n_other)."""
    feature_names = ["n_missense", "n_synonymous", "n_nonsense", "n_frameshift", "n_other"]

    # 결측치와 WT를 제외한 실제 변이만 추출, 한 칸에 여러 변이면 각각 분리
    mutations = X.stack().dropna()
    mutations = mutations[mutations != "WT"]
    tokens = mutations.astype(str).str.split().explode().dropna()

    # 변이 표기 패턴에 따라 유형 분류
    def classify(token):
        if "fs" in token.lower():
            return "n_frameshift"
        if re.fullmatch(r"[A-Z]\d+\*", token):
            return "n_nonsense"
        if re.fullmatch(r"[A-Z]\d+[A-Z]", token):
            if token[0] == token[-1]:
                return "n_synonymous"
            return "n_missense"
        return "n_other"

    types = tokens.map(classify)
    counts = pd.crosstab(tokens.index.get_level_values(0), types)
    return counts.reindex(index=X.index, columns=feature_names, fill_value=0).astype("int32")


def amino_acid_pair_counts(X: pd.DataFrame) -> pd.DataFrame:
    """환자별 아미노산 치환 쌍 개수 (예: V600E -> aa_V_E), 400개 고정 열."""
    amino_acids = "ACDEFGHIKLMNPQRSTVWY"
    pairs = [f"aa_{a}_{b}" for a in amino_acids for b in amino_acids]

    mutations = X.stack().dropna()
    mutations = mutations[mutations != "WT"]
    tokens = mutations.astype(str).str.split().explode().dropna()

    # 아미노산 치환 표기 추출 (같은 아미노산끼리는 제외)
    matches = tokens.str.extract(r"^([A-Z])\d+([A-Z])$")
    valid = (
        matches[0].isin(list(amino_acids))
        & matches[1].isin(list(amino_acids))
        & (matches[0] != matches[1])
    )
    matches = matches.loc[valid]
    pair_names = "aa_" + matches[0] + "_" + matches[1]

    counts = pd.crosstab(pair_names.index.get_level_values(0), pair_names)
    return counts.reindex(index=X.index, columns=pairs, fill_value=0).astype("int32")


def gene_truncation_flags(X: pd.DataFrame) -> pd.DataFrame:
    """유전자별 단백질 절단형 변이(Nonsense, Frameshift) 여부 0/1."""
    def has_truncation(value):
        if pd.isna(value):
            return 0
        tokens = str(value).split()
        return int(any(
            "fs" in token.lower() or re.fullmatch(r"[A-Z]\d+\*", token)
            for token in tokens
        ))

    result = X.apply(lambda col: col.map(has_truncation))
    result.columns = [f"trunc_{col}" for col in X.columns]
    return result.astype("int8")


def make_E(X: pd.DataFrame) -> pd.DataFrame:
    """E 피처 9,174열: 0/1 + log1p(변이 수) + log1p(변이 유형 5개) + log1p(아미노산 쌍 400개) + 유전자별 끊김형."""
    parts = [binarize_mutations(X), np.log1p(mutation_count(X)),
             np.log1p(mutation_type_counts(X)), np.log1p(amino_acid_pair_counts(X)),
             gene_truncation_flags(X)]
    out = pd.concat(parts, axis=1).astype(np.float32)
    assert not out.columns.has_duplicates and out.index.equals(X.index)
    return out
# ===== 【② 피처】 끝 =====

## 5. 【③ 모델】 모델 정의 · 담당 ③ 안주형
최종 모델과 하이퍼파라미터를 숫자로 적어요. 현재: 08 가중치 앙상블 (가중치 XGBoost E + LR E, LB 0.3687)

In [ ]:
# ===== 【③ 모델】 시작 =====
# 출처: notebooks/A_model/06 (가중치 XGBoost E + LR E), 제출 08 (LB 0.3687)
from scipy.sparse import csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

N_CLASSES = 26
XGB_PARAMS = dict(n_estimators=250, max_depth=3, learning_rate=0.05, subsample=0.8,
                  colsample_bytree=0.8, objective="multi:softprob", num_class=N_CLASSES,
                  eval_metric="mlogloss", tree_method="hist", device="cpu",
                  random_state=SEED, n_jobs=-1)
LR_PARAMS = dict(C=0.1, class_weight="balanced", max_iter=3000, solver="lbfgs")


class InvSqrtWeightedXGB(XGBClassifier):
    """클래스 가중치 1/√(클래스별 환자 수), 학습에 들어온 y로만 계산."""
    def fit(self, X, y, **kwargs):
        y = np.asarray(y, dtype=int)
        counts = np.bincount(y, minlength=N_CLASSES)
        w = 1.0 / np.sqrt(counts[y])
        return super().fit(X, y, sample_weight=(w / w.mean()).astype(np.float32), **kwargs)


def to_sparse_E(X):
    """LR 입력: E 피처를 희소행렬로 변환."""
    return csr_matrix(make_E(X).to_numpy())


def make_xgb():
    return Pipeline([("features", FunctionTransformer(make_E)),
                     ("model", InvSqrtWeightedXGB(**XGB_PARAMS))])


def make_lr():
    return Pipeline([("features", FunctionTransformer(to_sparse_E)),
                     ("model", LogisticRegression(**LR_PARAMS))])


MODELS = {"xgb_E_weighted": make_xgb, "lr_E": make_lr}
# ===== 【③ 모델】 끝 =====

## 6. (선택) 교차검증 · 공통 검증 방식과 같아요

In [ ]:
from sklearn.base import clone
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

def run_cv(model, X, y):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    scores = []
    for tr, va in skf.split(np.zeros(len(y)), y):
        m = clone(model).fit(X.iloc[tr], y[tr])           # fold의 학습 부분에만 fit
        scores.append(f1_score(y[va], m.predict(X.iloc[va]), average="macro"))
    print(f"CV Macro F1 = {np.mean(scores):.4f} (std {np.std(scores):.4f})")
    return scores

if RUN_CV:
    for name, make in MODELS.items():
        print(name); run_cv(make(), X, y)

## 7. 전체 train으로 학습 → test 예측 확률

In [ ]:
test_probas = {}
for name, make in MODELS.items():
    model = make().fit(X, y)                 # train 전체에만 fit
    test_probas[name] = model.predict_proba(X_test)
    print(name, "학습 완료")

## 8. 【④ 경량·앙상블】 최종 예측 결합 · 담당 ④ 박유정
모델이 하나면 그대로, 여러 개면 정한 가중치로 평균해요. 현재: 50:50 (08 기준, notebooks/A_model/06)

In [ ]:
# ===== 【④ 경량·앙상블】 시작 =====
ENSEMBLE_WEIGHTS = {"xgb_E_weighted": 0.5, "lr_E": 0.5}   # 08 가중치 앙상블 (LB 0.3687)

names = list(test_probas)
w = (np.ones(len(names)) / len(names) if ENSEMBLE_WEIGHTS is None
     else np.array([ENSEMBLE_WEIGHTS[n] for n in names]) / sum(ENSEMBLE_WEIGHTS.values()))
final_proba = sum(wi * test_probas[n] for wi, n in zip(w, names))
# ===== 【④ 경량·앙상블】 끝 =====

## 9. 제출 파일 저장

In [ ]:
sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
assert len(sub) == len(test) and (sub[ID_COL].values == test[ID_COL].values).all()
sub[TARGET] = le.inverse_transform(final_proba.argmax(axis=1))
sub.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("저장:", OUTPUT_PATH, sub.shape)
sub.head()